In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV, RandomizedSearchCV
from sklearn.metrics import make_scorer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [2]:
# Dosya yolu
df = pd.read_csv("all_data_ratio.csv")

In [5]:
# Feature map
feature_map = {
    "uzman": ["yıl", "nufus", "hastane", "yatak", "asistan", "pratisyen", "hemsire", "ebe", "diger"],
    "pratisyen": ["yıl", "nufus", "hastane", "yatak", "asistan", "uzman", "hemsire", "ebe", "diger"],
    "asistan": ["yıl", "nufus", "hastane", "yatak", "uzman", "pratisyen", "hemsire", "ebe", "diger"],
    "ebe": ["yıl", "nufus", "hastane", "yatak", "asistan", "uzman", "hemsire", "pratisyen", "diger"],
    "diger": ["yıl", "nufus", "hastane", "yatak", "asistan", "pratisyen", "hemsire", "ebe", "uzman"],
    "hemsire": ["yıl", "nufus", "hastane", "yatak", "asistan", "uzman", "pratisyen", "ebe", "diger"],
    "yatak": ["yıl", "nufus", "hastane", "hemsire", "asistan", "uzman", "pratisyen", "ebe", "diger"],
    "hastane": ["yıl", "nufus", "yatak", "hemsire", "asistan", "uzman", "pratisyen", "ebe", "diger"]
}

In [7]:
tahminler = []
tscv = TimeSeriesSplit(n_splits=2)
mae_scorer = make_scorer(mean_absolute_error, greater_is_better=False)

for il in df["il_adı"].unique():
    df_il = df[df["il_adı"] == il]
    if df_il[df_il["yıl"] <= 2021].shape[0] < 5:
        continue

    for hedef, features in feature_map.items():
        if hedef not in df_il.columns or any(f not in df_il.columns for f in features):
            continue

        df_target = df_il[features + [hedef]].dropna()
        if df_target.empty or df_target[df_target["yıl"] <= 2021].shape[0] == 0:
            continue

        train_data = df_target[df_target['yıl'] <= 2021]
        val_data = df_target[df_target['yıl'] == 2022]
        test_data = df_target[df_target['yıl'] == 2023]

        if val_data.empty or test_data.empty:
            continue

        X_train = train_data[features]
        y_train = train_data[hedef]
        X_val = val_data[features]
        y_val = val_data[hedef]
        X_test = test_data[features]
        y_test = test_data[hedef]

        if len(X_train) < 10 or len(set(y_train)) <= 1:
            continue

        # Random Forest
        try:
            rf_params = {'n_estimators': [50, 100], 'max_depth': [None, 5]}
            rf_grid = GridSearchCV(RandomForestRegressor(random_state=42), rf_params, cv=tscv,
                                   n_jobs=-1, scoring=mae_scorer)
            rf_grid.fit(X_train, y_train)
            rf = rf_grid.best_estimator_
        except:
            rf = RandomForestRegressor(n_estimators=100, random_state=42).fit(X_train, y_train)

        # Extra Trees
        try:
            et_params = {'n_estimators': [50, 100], 'max_depth': [None, 5]}
            et_grid = GridSearchCV(ExtraTreesRegressor(random_state=42), et_params, cv=tscv,
                                   n_jobs=-1, scoring=mae_scorer)
            et_grid.fit(X_train, y_train)
            et = et_grid.best_estimator_
        except:
            et = ExtraTreesRegressor(n_estimators=100, random_state=42).fit(X_train, y_train)

        # Gradient Boosting
        try:
            gb_params = {'n_estimators': [50, 100], 'learning_rate': [0.05, 0.1], 'max_depth': [3, 5]}
            gb_grid = GridSearchCV(GradientBoostingRegressor(random_state=42), gb_params, cv=tscv,
                                   n_jobs=-1, scoring=mae_scorer)
            gb_grid.fit(X_train, y_train)
            gb = gb_grid.best_estimator_
        except:
            gb = GradientBoostingRegressor(n_estimators=100, random_state=42).fit(X_train, y_train)

        # Tahminler
        y_val_pred_rf = rf.predict(X_val)[0]
        y_val_pred_et = et.predict(X_val)[0]
        y_val_pred_gb = gb.predict(X_val)[0]

        y_test_pred_rf = rf.predict(X_test)[0]
        y_test_pred_et = et.predict(X_test)[0]
        y_test_pred_gb = gb.predict(X_test)[0]

        tahminler.append({
            "il_adı": il,
            "hedef": hedef,
            "gercek_2022": y_val.values[0],
            "tahmin_2022_rf": y_val_pred_rf,
            "tahmin_2022_et": y_val_pred_et,
            "tahmin_2022_gb": y_val_pred_gb,
            "gercek_2023": y_test.values[0],
            "tahmin_2023_rf": y_test_pred_rf,
            "tahmin_2023_et": y_test_pred_et,
            "tahmin_2023_gb": y_test_pred_gb
        })


In [9]:
df_tahmin = pd.DataFrame(tahminler)

In [11]:
# Doğruluk oranı hesaplama (2023 yılı için)
for model in ["rf", "et", "gb"]:
    tahmin_kolonu = f"tahmin_2023_{model}"
    df_tahmin[f"{model}_yuzde_dog"] = 100 - (
        (np.abs(df_tahmin["gercek_2023"] - df_tahmin[tahmin_kolonu]) / df_tahmin["gercek_2023"]) * 100
    )
    # Değerleri 0–100 aralığına kısıtla
    df_tahmin[f"{model}_yuzde_dog"] = df_tahmin[f"{model}_yuzde_dog"].clip(lower=0, upper=100)

print("\n📊 Şehir Bazlı Ortalama Doğruluklar (2023):")
print("-" * 40)
for model in ["rf", "et", "gb"]:
    ortalama = df_tahmin[f"{model}_yuzde_dog"].mean()
    print(f" {model.upper():<4} doğruluk ort.: {ortalama:.2f}%")

# Tahmin detayları içeren DataFrame'i CSV'ye yaz
df_tahmin.to_csv("sehir_bazli_2023_tahmin_dogruluk_3model.csv", index=False)
print("\n✅ Tahmin doğrulukları 'sehir_bazli_2023_tahmin_dogruluk_3model.csv' olarak kaydedildi.")


📊 Şehir Bazlı Ortalama Doğruluklar (2023):
----------------------------------------
 RF   doğruluk ort.: 78.87%
 ET   doğruluk ort.: 78.92%
 GB   doğruluk ort.: 78.41%

✅ Tahmin doğrulukları 'sehir_bazli_2023_tahmin_dogruluk_3model.csv' olarak kaydedildi.


In [13]:
print("\n📉 Model Performans Metrikleri (2023):")
print("{:<6} | {:>10} | {:>10} | {:>10}".format("Model", "MAE", "RMSE", "R²"))
print("-" * 44)

metric_sonuclar = []

for model in ["rf", "et", "gb"]:
    y_true = df_tahmin["gercek_2023"]
    y_pred = df_tahmin[f"tahmin_2023_{model}"]

    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred, squared=False)
    r2 = r2_score(y_true, y_pred)

    print("{:<6} | {:>10.2f} | {:>10.2f} | {:>10.3f}".format(model.upper(), mae, rmse, r2))

    metric_sonuclar.append({
        "Model": model.upper(),
        "MAE": round(mae, 2),
        "RMSE": round(rmse, 2),
        "R2": round(r2, 3)
    })

# Metrikleri CSV olarak kaydet
df_metrics = pd.DataFrame(metric_sonuclar)
df_metrics.to_csv("sehir_bazli_2023_model_metrikleri.csv", index=False)
print("\n✅ Performans metrikleri 'sehir_bazli_2023_model_metrikleri.csv' dosyasına kaydedildi.")


📉 Model Performans Metrikleri (2023):
Model  |        MAE |       RMSE |         R²
--------------------------------------------
RF     |     297.69 |    1052.59 |      0.915
ET     |     297.21 |    1095.41 |      0.908
GB     |     301.13 |    1154.20 |      0.898

✅ Performans metrikleri 'sehir_bazli_2023_model_metrikleri.csv' dosyasına kaydedildi.


C:\Users\Bedirhan\anaconda3\Lib\site-packages\sklearn\metrics\_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
C:\Users\Bedirhan\anaconda3\Lib\site-packages\sklearn\metrics\_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
C:\Users\Bedirhan\anaconda3\Lib\site-packages\sklearn\metrics\_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


In [15]:
# Türkiye'nin 7 coğrafi bölgesi
bolge_haritasi = {
    'Marmara': ["İstanbul", "Edirne", "Kırklareli", "Tekirdağ", "Kocaeli", "Sakarya", "Yalova", "Bilecik", "Balıkesir", "Çanakkale", "Bursa"],
    'Ege': ["İzmir", "Aydın", "Muğla", "Manisa", "Afyonkarahisar", "Uşak", "Kütahya", "Denizli"],
    'Akdeniz': ["Antalya", "Mersin", "Adana", "Hatay", "Kahramanmaraş", "Osmaniye", "Isparta", "Burdur"],
    'İç Anadolu': ["Ankara", "Konya", "Kayseri", "Eskişehir", "Nevşehir", "Aksaray", "Kırıkkale", "Niğde", "Karaman", "Sivas", "Yozgat", "Kırşehir", "Çankırı"],
    'Karadeniz': ["Samsun", "Trabzon", "Ordu", "Giresun", "Rize", "Artvin", "Tokat", "Amasya", "Bartın", "Zonguldak", "Karabük", "Sinop", "Çorum", "Bayburt", "Gümüşhane", "Kastamonu", "Bolu", "Düzce"],
    'Doğu Anadolu': ["Erzurum", "Van", "Malatya", "Elazığ", "Kars", "Ağrı", "Bitlis", "Iğdır", "Tunceli", "Ardahan", "Erzincan", "Bingöl", "Muş", "Hakkari"],
    'Güneydoğu Anadolu': ["Diyarbakır", "Gaziantep", "Şanlıurfa", "Mardin", "Batman", "Siirt", "Şırnak", "Adıyaman", "Kilis"]
}

def bolge_bul(il):
    for bolge, iller in bolge_haritasi.items():
        if il in iller:
            return bolge
    return "Bilinmiyor"

# Bölge sütununu ekle
df_tahmin["bolge"] = df_tahmin["il_adı"].apply(bolge_bul)

# Bölgelere göre grupla ve toplam değerleri al
bolgesel = df_tahmin.groupby(["bolge", "hedef"])[[
    "gercek_2023",
    "tahmin_2023_rf",
    "tahmin_2023_et",
    "tahmin_2023_gb"
]].sum().reset_index()

# CSV olarak kaydet
bolgesel.to_csv("bolge_bazli_2023_tahmin3model.csv", index=False)
print("✅ 'bolge_bazli_2023_tahmin3model.csv' dosyası kaydedildi.")

✅ 'bolge_bazli_2023_tahmin3model.csv' dosyası kaydedildi.


In [17]:
# 1. Dosyayı oku
df = pd.read_csv("sehir_bazli_2023_tahmin_dogruluk_3model.csv")  # Gerekirse dosya adını kontrol et

# 2. Doğruluk hesaplama fonksiyonu
def hesapla_dog(model_etiketi):
    df[f"{model_etiketi}_yuzde_dog"] = 100 - (
        np.abs(df["gercek_2023"] - df[f"tahmin_2023_{model_etiketi}"]) / df["gercek_2023"] * 100
    )
    df[f"{model_etiketi}_yuzde_dog"] = df[f"{model_etiketi}_yuzde_dog"].clip(0, 100)

# 3. Üç model için hesapla
for model in ["rf", "et", "gb"]:
    hesapla_dog(model)

# 4. Ortalamaları yazdır
print("\nŞehir Bazlı Doğruluk Ortalamaları (2023):")
for model in ["rf", "et", "gb"]:
    ortalama = round(df[f"{model}_yuzde_dog"].mean(), 2)
    print(f" - {model.upper()}: {ortalama}%")

# 5. Güncel dosya olarak kaydet
df.to_csv("sehir_tahmin_yuzdeler_3model.csv", index=False)
print("\n✅ Güncel doğruluk yüzdeleri 'sehir_tahmin_yuzdeler_3model.csv' olarak kaydedildi.")


Şehir Bazlı Doğruluk Ortalamaları (2023):
 - RF: 78.87%
 - ET: 78.92%
 - GB: 78.41%

✅ Güncel doğruluk yüzdeleri 'sehir_tahmin_yuzdeler_3model.csv' olarak kaydedildi.


In [23]:
# 1. Dosyayı oku
df = pd.read_csv("bolge_bazli_2023_tahmin3model.csv")  # Gerekirse dosya adını değiştir

# 2. Her model için yüzdelik doğruluk hesapla
for model in ["rf", "et", "gb"]:
    df[f"{model}_yuzde_dog"] = 100 - (
        np.abs(df["gercek_2023"] - df[f"tahmin_2023_{model}"]) / df["gercek_2023"] * 100
    )
    df[f"{model}_yuzde_dog"] = df[f"{model}_yuzde_dog"].clip(0, 100)

# 3. Ortalamaları hesapla ve yazdır
print(f"\n📍 Bölge Doğruluk Ortalamaları (2023):")
for model in ["rf", "et", "gb"]:
    ortalama = round(df[f"{model}_yuzde_dog"].mean(), 2)
    print(f" - {model.upper()}: {ortalama}%")

# 4. Dosyaya kaydet
df.to_csv("bolge_tahmin_yuzdeler_3model.csv", index=False)
print("Bölge bazlı doğruluk yüzdeleri 'bolge_tahmin_yuzdeler_3model.csv' olarak kaydedildi.")


📍 Bölge Doğruluk Ortalamaları (2023):
 - RF: 84.88%
 - ET: 85.28%
 - GB: 84.61%
Bölge bazlı doğruluk yüzdeleri 'bolge_tahmin_yuzdeler_3model.csv' olarak kaydedildi.


In [3]:
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 1. Dosyayı oku
df = pd.read_csv("bolge_bazli_2023_tahmin3model.csv")

# 2. Modelleri sırala
models = ["rf", "et", "gb"]
metrics = []

# 3. Her model için hata metriklerini hesapla
for model in models:
    y_true = df["gercek_2023"]
    y_pred = df[f"tahmin_2023_{model}"]

    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred, squared=False)
    r2 = r2_score(y_true, y_pred)

    metrics.append({
        "Model": model.upper(),
        "MAE": round(mae, 2),
        "RMSE": round(rmse, 2),
        "R2": round(r2, 3)
    })

# 4. DataFrame'e dönüştür ve yazdır
df_metrics = pd.DataFrame(metrics)
print("\n📊 Bölge Bazlı 3 Model Performans Metrikleri (2023):")
print(df_metrics.to_string(index=False))

# 5. CSV olarak kaydet
df_metrics.to_csv("bolge_bazli_2023_model_metrikleri_3model.csv", index=False)
print("\n✅ Metrikler 'bolge_bazli_2023_model_metrikleri_3model.csv' dosyasına kaydedildi.")



📊 Bölge Bazlı 3 Model Performans Metrikleri (2023):
Model     MAE    RMSE    R2
   RF 3142.82 5604.91 0.895
   ET 3107.30 5695.60 0.892
   GB 3175.84 5853.05 0.886

✅ Metrikler 'bolge_bazli_2023_model_metrikleri_3model.csv' dosyasına kaydedildi.


C:\Users\Bedirhan\anaconda3\Lib\site-packages\sklearn\metrics\_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
C:\Users\Bedirhan\anaconda3\Lib\site-packages\sklearn\metrics\_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
C:\Users\Bedirhan\anaconda3\Lib\site-packages\sklearn\metrics\_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


In [25]:
import warnings
warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np

# 1. Tahmin çıktısını oku
df = pd.read_csv("sehir_bazli_2023_tahmin_dogruluk_3model.csv")

# 2. En iyi 3 modeli doğruluk ortalamasına göre belirle
model_list = ["rf", "et", "gb"]
dog_list = [(model, df[f"{model}_yuzde_dog"].mean()) for model in model_list]
dog_list_sorted = sorted(dog_list, key=lambda x: x[1], reverse=True)

# İlk 3 modeli seç
en_iyi_uc = [m[0] for m in dog_list_sorted[:3]]
print("\n✅ En iyi 3 model:", en_iyi_uc)

# 3. Ağırlık kombinasyonlarını oluştur
grid = []
for w1 in np.arange(0.1, 1.0, 0.1):
    for w2 in np.arange(0.1, 1.0 - w1, 0.1):
        w3 = 1.0 - w1 - w2
        if w3 > 0:
            grid.append((round(w1, 2), round(w2, 2), round(w3, 2)))

en_iyi_agirlik = None
en_iyi_ortalama = 0

# 4. Her ağırlık kombinasyonu için doğruluk hesapla
for w in grid:
    ensemble_tahmin = (
        df[f"tahmin_2023_{en_iyi_uc[0]}"] * w[0] +
        df[f"tahmin_2023_{en_iyi_uc[1]}"] * w[1] +
        df[f"tahmin_2023_{en_iyi_uc[2]}"] * w[2]
    )
    ensemble_dog = 100 - (np.abs(df["gercek_2023"] - ensemble_tahmin) / df["gercek_2023"] * 100)
    ensemble_dog = ensemble_dog.clip(0, 100)
    ort = ensemble_dog.mean()

    if ort > en_iyi_ortalama:
        en_iyi_ortalama = ort
        en_iyi_agirlik = w
        df["tahmin_2023_ensemble"] = ensemble_tahmin
        df["ensemble_yuzde_dog"] = ensemble_dog

# 5. Sonuçları yazdır
print(f"\n🎯 En iyi ağırlık kombinasyonu: {en_iyi_agirlik}")
print("📊 Ensemble doğruluk ortalaması (2023): {:.2f}%".format(en_iyi_ortalama))

# 6. Kaydet
df.to_csv("sehir_bazli_2023_ensemble_model.csv", index=False)
print("💾 Ensemble çıktısı 'sehir_bazli_2023_ensemble_model.csv' olarak kaydedildi.")


✅ En iyi 3 model: ['et', 'rf', 'gb']

🎯 En iyi ağırlık kombinasyonu: (0.6, 0.3, 0.1)
📊 Ensemble doğruluk ortalaması (2023): 78.92%
💾 Ensemble çıktısı 'sehir_bazli_2023_ensemble_model.csv' olarak kaydedildi.


In [27]:
import pandas as pd
import numpy as np

# 1. Dosyayı oku
df = pd.read_csv("sehir_tahmin_yuzdeler_3model.csv")

# 2. En iyi 3 modelin doğruluk ortalamalarına göre seçimi
modeller = ["rf", "et", "gb"]
ortalamalar = {model: df[f"{model}_yuzde_dog"].mean() for model in modeller}
en_iyi3 = sorted(ortalamalar.items(), key=lambda x: x[1], reverse=True)[:3]

mod_adlari = [m[0] for m in en_iyi3]
print(f"\n🔍 Seçilen modeller: {mod_adlari}")

# 3. Sabit ağırlıklar (örnek olarak)
agirliklar = (0.5, 0.3, 0.2)  # mod_adlari sırasına göre

# 4. Ensemble tahmini oluştur
df["tahmin_2023_ensemble"] = (
    df[f"tahmin_2023_{mod_adlari[0]}"] * agirliklar[0] +
    df[f"tahmin_2023_{mod_adlari[1]}"] * agirliklar[1] +
    df[f"tahmin_2023_{mod_adlari[2]}"] * agirliklar[2]
)

# 5. Ensemble doğruluğu hesapla
df["ensemble_yuzde_dog"] = 100 - (
    np.abs(df["gercek_2023"] - df["tahmin_2023_ensemble"]) / df["gercek_2023"] * 100
)
df["ensemble_yuzde_dog"] = df["ensemble_yuzde_dog"].clip(0, 100)

# 6. Ortalamayı yazdır
ortalama = df["ensemble_yuzde_dog"].mean()
print(f"\n📊 Sabit Ağırlıklı Ensemble Doğruluk Ortalaması: {ortalama:.2f}%")

# 7. CSV kaydı
df.to_csv("sehir_tahmin_yuzdeler_ensemble.csv", index=False)
print("💾 Dosya 'sehir_tahmin_yuzdeler_ensemble.csv' olarak kaydedildi.")


🔍 Seçilen modeller: ['et', 'rf', 'gb']

📊 Sabit Ağırlıklı Ensemble Doğruluk Ortalaması: 78.89%
💾 Dosya 'sehir_tahmin_yuzdeler_ensemble.csv' olarak kaydedildi.


In [29]:
import pandas as pd
import numpy as np

# 1. Dosyayı oku
df = pd.read_csv("sehir_tahmin_yuzdeler_ensemble.csv")  # ensemble doğruluk içeren dosya

# 2. Bölge haritası
bolge_haritasi = {
    'Marmara': ["İstanbul", "Edirne", "Kırklareli", "Tekirdağ", "Kocaeli", "Sakarya", "Yalova", "Bilecik", "Balıkesir", "Çanakkale", "Bursa"],
    'Ege': ["İzmir", "Aydın", "Muğla", "Manisa", "Afyonkarahisar", "Uşak", "Kütahya", "Denizli"],
    'Akdeniz': ["Antalya", "Mersin", "Adana", "Hatay", "Kahramanmaraş", "Osmaniye", "Isparta", "Burdur"],
    'İç Anadolu': ["Ankara", "Konya", "Kayseri", "Eskişehir", "Nevşehir", "Aksaray", "Kırıkkale", "Niğde", "Karaman", "Sivas", "Yozgat", "Kırşehir", "Çankırı"],
    'Karadeniz': ["Samsun", "Trabzon", "Ordu", "Giresun", "Rize", "Artvin", "Tokat", "Amasya", "Bartın", "Zonguldak", "Karabük", "Sinop", "Çorum", "Bayburt", "Gümüşhane", "Kastamonu", "Bolu", "Düzce"],
    'Doğu Anadolu': ["Erzurum", "Van", "Malatya", "Elazığ", "Kars", "Ağrı", "Bitlis", "Iğdır", "Tunceli", "Ardahan", "Erzincan", "Bingöl", "Muş", "Hakkari"],
    'Güneydoğu Anadolu': ["Diyarbakır", "Gaziantep", "Şanlıurfa", "Mardin", "Batman", "Siirt", "Şırnak", "Adıyaman", "Kilis"]
}

def bolge_bul(il):
    for bolge, iller in bolge_haritasi.items():
        if il in iller:
            return bolge
    return "Bilinmiyor"

# 3. Bölge bilgisi ekle
df["bolge"] = df["il_adı"].apply(bolge_bul)

# 4. Bölge ve hedef bazında grupla, tahmin ve gerçekleri topla
bolge_df = df.groupby(["bolge", "hedef"])[["gercek_2023", "tahmin_2023_ensemble"]].sum().reset_index()

# 5. Ensemble doğruluğunu hesapla
bolge_df["ensemble_yuzde_dog"] = 100 - (
    np.abs(bolge_df["gercek_2023"] - bolge_df["tahmin_2023_ensemble"]) / bolge_df["gercek_2023"] * 100
)
bolge_df["ensemble_yuzde_dog"] = bolge_df["ensemble_yuzde_dog"].clip(0, 100)

# 6. Ortalama doğruluğu yazdır
ortalama = round(bolge_df["ensemble_yuzde_dog"].mean(), 2)
print(f"\n📍 Ensemble Bölge Doğruluk Ortalaması: {ortalama}%")

# 7. CSV olarak kaydet
bolge_df.to_csv("bolge_tahmin_yuzdeler_ensemble.csv", index=False)
print("💾 'bolge_tahmin_yuzdeler_ensemble.csv' dosyası kaydedildi.")


📍 Ensemble Bölge Doğruluk Ortalaması: 85.04%
💾 'bolge_tahmin_yuzdeler_ensemble.csv' dosyası kaydedildi.


In [1]:
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 1. CSV dosyasını oku
df = pd.read_csv("bolge_tahmin_yuzdeler_ensemble.csv")

# 2. Gerçek ve tahmin sütunlarını al
y_true = df["gercek_2023"]
y_pred = df["tahmin_2023_ensemble"]

# 3. Hata metriklerini hesapla
mae = mean_absolute_error(y_true, y_pred)
rmse = mean_squared_error(y_true, y_pred, squared=False)
r2 = r2_score(y_true, y_pred)

# 4. Sonuçları tablo formatına getir
metrics = pd.DataFrame([{
    "Model": "ENSEMBLE",
    "MAE": round(mae, 2),
    "RMSE": round(rmse, 2),
    "R2": round(r2, 3)
}])

# 5. Yazdır ve kaydet
print("\n📊 Ensemble Bölgesel Performans Metrikleri (2023):")
print(metrics.to_string(index=False))

metrics.to_csv("bolgesel_ensemble_2023_model_metrikleri.csv", index=False)
print("\n✅ Metrikler 'bolgesel_ensemble_2023_model_metrikleri.csv' dosyasına kaydedildi.")



📊 Ensemble Bölgesel Performans Metrikleri (2023):
   Model     MAE    RMSE    R2
ENSEMBLE 3128.03 5695.98 0.892

✅ Metrikler 'bolgesel_ensemble_2023_model_metrikleri.csv' dosyasına kaydedildi.


C:\Users\Bedirhan\anaconda3\Lib\site-packages\sklearn\metrics\_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
